# QQQ/SMH原始5分钟成交额：同bar交易均价语义门

冻结Futu `NONE` K_5M QQQ/SMH原件与NAS逐件SHA、Qlib同行5min平面。核原报告turnover/volume在原价OHLC同bar高低±1bp之内的比例、全年覆盖和相对OHLC4的残差；不读取Stage1未来标签或拟合。旧SOXL/SOXX 1m源语义失败不能自动外推到5m，5m也不因高流动性被默认合格。

In [1]:
import os,json,hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
root=Path.cwd();card=root/'research/stage1_v5_peer_5m_turnover_source_20261030';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['source_manifest'])==plan['source_manifest_sha256'] and sha(root/plan['nas_binding'])==plan['nas_binding_sha256'] and sha(root/plan['qlib_provider_uri']/'qlib_store_manifest.json')==plan['qlib_provider_manifest_sha256']
assert json.loads((card/'nas_original_readback.json').read_text())['status']=='passed'
binding=json.loads((root/plan['nas_binding']).read_text());assert binding['verified'] and binding['source_id']==plan['source_manifest_sha256']
qlib.init(provider_uri=str(root/plan['qlib_provider_uri']),region='us',kernels=1)
raw={}
for symbol,spec in plan['original_parquets'].items():
 path=root/spec['relative_path'];assert sha(path)==spec['sha256']
 x=pd.read_parquet(path,columns=['time_key','open','high','low','close','volume','turnover']);assert len(x)==spec['rows'];x['datetime']=pd.to_datetime(x.time_key)
 raw[symbol]=x.loc[x.datetime.between('2018-10-15 09:35:00','2022-12-22 16:00:00')].set_index('datetime').sort_index();assert raw[symbol].index.is_unique
print('original/NAS source identity passed',{k:len(v) for k,v in raw.items()})

[54191:MainThread](2026-10-30 17:41:19,392) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[54191:MainThread](2026-10-30 17:41:19,799) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[54191:MainThread](2026-10-30 17:41:19,799) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/peer_5min')}


original/NAS source identity passed {'QQQ': 82044, 'SMH': 82044}


In [2]:
bars=D.features(['QQQ','SMH'],['$open','$high','$low','$close','$volume','$price_valid'],start_time='2018-10-15 09:35:00',end_time='2022-12-22 16:00:00',freq='5min')
annual=[];summary={};maximum_volume_error=0.;maximum_price_error=0.
for symbol in ('QQQ','SMH'):
 q=bars.xs(symbol,level='instrument');x=raw[symbol];assert q.index.equals(x.index)
 volume_error=float(np.nanmax(np.abs(x.volume.to_numpy(float)-q['$volume'].to_numpy(float))));maximum_volume_error=max(maximum_volume_error,volume_error)
 for field in ('open','high','low','close'):maximum_price_error=max(maximum_price_error,float(np.nanmax(np.abs(x[field].to_numpy(float)-q['$'+field].to_numpy(float)))))
 z=x.copy();z['price_valid']=q['$price_valid'].to_numpy(float);z['year']=z.index.year;z['trade_date']=z.index.normalize()
 positive=z.price_valid.eq(1)&z.volume.gt(0)
 positive_fraction=float(z.loc[positive,'turnover'].gt(0).mean())
 v=z.loc[positive&z.turnover.gt(0)].copy();v['implied_vwap']=v.turnover/v.volume;v['ohlc4']=(v.open+v.high+v.low+v.close)/4
 v['inside_1bp']=(v.implied_vwap>=v.low*np.exp(-.0001))&(v.implied_vwap<=v.high*np.exp(.0001));v['residual_bp']=10000*np.log(v.implied_vwap/v.ohlc4)
 inside=float(v.inside_1bp.mean())
 perday=v.groupby(['trade_date','year']).agg(bars=('implied_vwap','size'),p90_abs_residual_bp=('residual_bp',lambda x:float(np.quantile(np.abs(x),.9)))).reset_index()
 for year,g in perday.groupby('year'):
  annual.append({'symbol':symbol,'year':int(year),'source_days':len(g),'bars':int(g.bars.sum()),'novel_residual_days':int(g.p90_abs_residual_bp.gt(.25).sum()),'median_day_p90_abs_residual_bp':float(g.p90_abs_residual_bp.median()),'inside_ohlc_fraction':float(v.loc[v.year.eq(year),'inside_1bp'].mean())})
 summary[symbol]={'positive_turnover_fraction':positive_fraction,'same_bar_inside_ohlc_fraction':inside,'bars':len(v),'days':int(v.trade_date.nunique())}
assert maximum_volume_error<=plan['gates']['maximum_raw_volume_qlib_absolute_error']
a=pd.DataFrame(annual).sort_values(['symbol','year']);a.to_csv(card/'annual_source_summary.csv',index=False)
gate=plan['gates'];passed=all(summary[s]['positive_turnover_fraction']>=gate['minimum_positive_turnover_fraction_when_volume_positive'] and summary[s]['same_bar_inside_ohlc_fraction']>=gate['minimum_each_symbol_same_bar_implied_vwap_inside_raw_ohlc_fraction_with_1bp_tolerance'] and int(a.loc[(a.symbol.eq(s))&(a.year.eq(2018)),'source_days'].iloc[0])>=gate['minimum_2018_source_days'] and all(int(a.loc[(a.symbol.eq(s))&(a.year.eq(y)),'source_days'].iloc[0])>=gate['minimum_full_year_source_days'] for y in (2019,2020,2021,2022)) and sum(int(v) for v in a.loc[(a.symbol.eq(s))&a.year.isin(plan['discovery_years']),'novel_residual_days'])>=gate['minimum_discovery_days_with_daily_p90_abs_reported_vwap_minus_ohlc4_gt_0_25bp_each_symbol'] for s in ('QQQ','SMH'))
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'qlib_rows':len(bars),'raw_rows':sum(len(x) for x in raw.values()),'maximum_raw_volume_qlib_error':maximum_volume_error,'maximum_raw_ohlc_qlib_error':maximum_price_error,'symbol_summary':summary,'annual':annual,'source_gate_pass':bool(passed),'historical_first_receipt_qualified':False,'economic_values_accessed':False,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'peer_reported_5m_turnover_semantics_pass_needs_information' if passed else 'peer_reported_5m_turnover_semantics_failed_before_fit'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n');print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_peer_5m_turnover_source_20261030",
  "plan_sha256": "e28bb34ceadfadd74f1ad3b6cadf33fd1ef7e2e1ca8dcf340f94635232832eb4",
  "qlib_rows": 164088,
  "raw_rows": 164088,
  "maximum_raw_volume_qlib_error": 0.0,
  "maximum_raw_ohlc_qlib_error": 1.523437498462954e-05,
  "symbol_summary": {
    "QQQ": {
      "positive_turnover_fraction": 1.0,
      "same_bar_inside_ohlc_fraction": 0.9324573058497995,
      "bars": 82037,
      "days": 1056
    },
    "SMH": {
      "positive_turnover_fraction": 1.0,
      "same_bar_inside_ohlc_fraction": 0.997415768686918,
      "bars": 82036,
      "days": 1056
    }
  },
  "annual": [
    {
      "symbol": "QQQ",
      "year": 2018,
      "source_days": 53,
      "bars": 4062,
      "novel_residual_days": 53,
      "median_day_p90_abs_residual_bp": 7.356630780242401,
      "inside_ohlc_fraction": 0.9492860659773511
    },
    {
      "symbol": "QQQ",
      "year": 2019,
      "source_days": 252,
      "bars": 19548,
      "novel_

## 中文结论

SOXX历史Goal未通过，0新模型/账户。QQQ/SMH Futu NONE原始5分钟parquet各自本地/NAS SHA通过；2018–2022各1056日，共164088条原件与Qlib 5m同钟，原价成交量差0、OHLC仅浮点差1.52e−5，正成交量bar报告turnover均>0。以turnover/volume作为当bar成交均价，放宽1bp仍仅QQQ93.2457%、SMH99.7416%位于本bar原价高低范围，分别低于冻结99.9%必要门。发现期报告成交额/量与OHLC4的bp级残差有覆盖，但不补语义门；不按已见数据选高成交量bar或改容忍，来源版本在拟合前关闭。历史首次接收和成交过滤口径unknown。